# I want to walk you through this: from your threshold to a foundation-model *idea*

This is not a generic machine-learning class. I am not going to drop a classifier
in front of you and call it a day.

You have already done something concrete — a moving average and a fixed threshold
opening and closing a robotic arm. So I am not asking “can AI classify EMG?”
I am asking:

**Where does *your* controller break? How far can signal processing still carry
you? When do we actually need a *representation* that might survive a new user?**

The encoder I am using is **TinyMyo** (PulpBio / BioFoundation). I will freeze it
and train only a linear probe. One file I will **not** use: the UCI fine-tuned
weights. That checkpoint has already seen these gesture labels. It would make
the “foundation representation” story dishonest.

Please remember this: if TinyMyo does not beat your handcrafted features, **we
keep the number**. I will not swap datasets to make the story prettier.


## How I want you to run this in class

1. If the data is not ready yet: `python scripts/prepare.py` (needs the internet; skip if you already did it)
2. Leave `FAST_DEMO = True` — that is the classroom setting
3. Kernel: `yijun_emg_demo`. Then run top to bottom with me

I will tell you what I am asking before each block of code. Do not jump to the
accuracy number.

If your laptop is Intel CPU + Intel iGPU, read `运行说明.md` first.
We do not use the iGPU. `DEVICE cpu` is the correct printout.


In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from IPython import get_ipython
    _ipy = get_ipython()
    if _ipy is not None:
        _ipy.run_line_magic("matplotlib", "inline")
except Exception:
    pass

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

FAST_DEMO = True
SEED = 0
MAX_ITER = 200 if FAST_DEMO else 800

import importlib
import src.dataset
import src.evaluation
import src.machine_info
import src.plotting
import src.pretrained_encoder
import src.signal_processing
for _mod in (
    src.dataset,
    src.evaluation,
    src.machine_info,
    src.plotting,
    src.pretrained_encoder,
    src.signal_processing,
):
    importlib.reload(_mod)

from src.dataset import (
    CLASSIFICATION_CLASS_NAMES, UCI_RAW_CLASS_NAMES,
    collect_raw_class_snippets, extract_binary_span, load_fast_demo,
    load_meta, load_raw_example, load_windowed, processed_paths,
    random_window_split, subject_split_indices,
)
from src.evaluation import (
    binary_accuracy, few_label_curve, fit_eval, overlapping_subjects,
    pca_2d, threshold_predict,
)
from src.plotting import (
    plot_confusion, plot_crosstalk_diagram, plot_few_label_curves,
    plot_grouped_bars, plot_model_ab_table, plot_notch_before_after,
    plot_pca_scatter, plot_raw_channels, plot_raw_class_gallery,
    plot_summary_progression,
    plot_collection_photos, plot_four_dataset_channels,
    plot_three_pipelines, plot_tinymyo_mask_story,
    plot_threshold_controller, plot_threshold_sweep,
    plot_moving_average_as_lowpass, plot_time_fft_psd,
    plot_window_length_sweep, set_plot_language,
)
from src.machine_info import format_machine_report
from src.pretrained_encoder import (
    build_encoder, count_parameters, default_checkpoint_path,
    encode_windows, load_pretrained_weights, select_device,
)
from src.signal_processing import (
    add_synthetic_mains, fft_magnitude, mean_channel_envelope,
    notch_filter, peak_near, welch_psd,
)

font_name = set_plot_language("en")
DEVICE = select_device()
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("ROOT", ROOT)
print("FAST_DEMO", FAST_DEMO)
print("DEVICE", DEVICE)
print("plot language", font_name)
print(format_machine_report(lang="en"))


In [ ]:
meta = load_meta()
rec = load_raw_example(processed_paths()["example"])
paths = processed_paths()
if FAST_DEMO:
    demo = load_fast_demo()
    X, y, subject = demo["X"], demo["y"].astype(np.int32), demo["subject"].astype(np.int32)
    Z, F = demo["embeddings"], demo["features"]
    train_subjects = [int(s) for s in demo["train_subjects"]]
    test_subjects = [int(s) for s in demo["test_subjects"]]
    window_samples = int(demo["window_samples"])
else:
    windows = load_windowed(paths["windows"])
    X, y, subject = windows.X, windows.y, windows.subject
    Z = np.load(paths["embeddings"])
    F = np.load(ROOT / "cache" / "handcrafted_features.npz")["features"]
    train_subjects = meta["splits"]["train"]
    test_subjects = meta["splits"]["test"]
    window_samples = int(windows.window_samples)

fs = float(meta["uci_fs"])
pair = meta["binary_pair"]
if pair["kind"] == "rest_vs_fist":
    pair_title = "hand at rest vs fist"
else:
    pair_title = f"amplitude binary: {pair['low_name']} vs {pair['high_name']}"
class_names = CLASSIFICATION_CLASS_NAMES
print("window protocol:", meta["window_protocol"])
print("X", X.shape, "y", y.shape, "subjects", len(np.unique(subject)))
print("embeddings Z", Z.shape, "handcrafted F", F.shape)
print("binary-demo pair:", pair_title)


---
# Datasets — four corpora, not one muscle map

Classification uses **UCI 481**: Myo, 8 channels, 200 Hz, Lobachevsky University.
TinyMyo was pretrained on three other **2 kHz** sets: NinaPro **DB6**, **DB7**,
**EMG2Pose**. All EMG. **Not the same geometry.**

- **UCI / Myo:** eight equally spaced pods. **No official muscle-per-column map.**
  Lobov 2018 only says that 2/4/6/8 sit *near* FCR, extensors, ECU, FCU if worn well.
- **DB6:** 14 Delsys. Eight at the radio-humeral joint, six further distal.
- **DB7:** 12 Delsys. Eight in a ring, plus FDS/EDC, plus biceps/triceps.
- **EMG2Pose:** 16 bipolar wrist pods, ulnar gap, **channels 0 and 15 beside it**.

TinyMyo expects up to 16 channels. UCI has eight. That is one reason the probe
can lose.

Photos are from the official NinaPro site. I do not have the UCI paper’s
subject photo; I will not fake one.


In [ ]:
fig = plot_collection_photos(ROOT / "outputs" / "datasets_collection_photos.png")
display(fig)
plt.show()
fig = plot_four_dataset_channels(ROOT / "outputs" / "datasets_channel_maps.png")
display(fig)
plt.show()


---
# Foundation model — I will tell you TinyMyo’s homework first

Not a bigger classifier. Pretraining had **none** of today’s six UCI labels.
It ate the three 2 kHz sets above. Weights: `TinyMyo.safetensors`. Not the
UCI fine-tune.

Original signal: ≤16 channels × 1000 samples (0.5 s). Cover about half of the
10 ms scraps, put 20 voltages back. Then throw the reconstruction head away.
We freeze the encoder. We only ask whether gestures cut.

Misaligned channels and a 10× rate gap are enough to lose to your 32 numbers.
That is the lesson, not an accident.


In [ ]:
fig_mask = plot_tinymyo_mask_story(ROOT / "outputs" / "exp10_mask_story.png")
display(fig_mask)
plt.show()


---
# Act I — What I would have done first

If you handed me EMG tomorrow, I would reach for amplitude, averaging, and
filters. **Can I clean my way out of this?** Let’s try together.


## Experiment 1 — Look at the raw EMG with me

I want a naive question on the table first: if you only watch amplitude, can
you tell what the person intends?

Surface EMG is not one number. It is a **multichannel** voltage sitting on the
skin, changing with time. Every electrode sees a mixture of nearby muscles.

The dataset is UCI EMG gestures (id 481). TinyMyo’s script treats it as
**200 Hz**. Here is the official codebook — keep it next to the plots:

| code | meaning |
| --- | --- |
| 0 | unmarked |
| 1 | hand at rest |
| 2 | hand clenched in a fist |
| 3 | wrist flexion |
| 4 | wrist extension |
| 5 | radial deviations |
| 6 | ulnar deviations |
| 7 | extended palm (not everyone did this) |

I need you to lock this in: **unmarked is not rest.** Rest is 1. Zero just means
“this stretch was not labelled.”

I will show you a continuous recording first, then **one held snippet per code
0–7**. Palm is missing for most people. If this example file has no 7, I will
borrow a snippet from subject 11 or 30 — the people the official notes say
actually did palm. I will write the other subject on the title. I will not
pretend it is the same person.


In [ ]:
from src.dataset import collect_raw_class_snippets
from src.plotting import plot_raw_class_gallery

t = rec.time_ms / 1000.0
sl = slice(0, int(min(len(t), 8 * fs)))
print("example recording:", rec.source_path)
print("subject", rec.subject, "shape (time, ch)", rec.emg.shape, "fs", rec.fs)
print("raw labels in this file:", sorted(int(x) for x in np.unique(rec.raw_label)))
print("classification windows X", X.shape, "n_classes", len(np.unique(y)), "(unmarked and palm dropped)")
fig = plot_raw_channels(
    t[sl], rec.emg[sl], rec.raw_label[sl], channels=(0, 1, 2, 3),
    title=f"Continuous record (first seconds) — subject {rec.subject}",
)
plt.show()

snippets = collect_raw_class_snippets(rec, labels=range(8), max_seconds=3.0)
print("codes found:", sorted(snippets))
for code in range(8):
    snip = snippets.get(code)
    name = UCI_RAW_CLASS_NAMES[code]
    if snip is None:
        print(f"  {code} {name}: not found")
        continue
    src = "this file" if snip["from_primary"] else f"subject {snip['subject']}"
    print(f"  {code} {name}: {snip['n_samples']} samples from {src}")
fig = plot_raw_class_gallery(
    snippets, UCI_RAW_CLASS_NAMES, channels=(0, 1, 2, 3),
    title="Official codes 0–7: one hold each (channels 1–4)",
)
plt.show()


After the figure: if two movements live in the same amplitude range, can your
fixed threshold still be trusted?


## Experiment 2 — I am rebuilding *your* threshold controller

Rectify, moving-average, one line. That is your old switch. I want to know:
**can it still be an on/off controller?**

Rectification folds the negative voltages up. The moving average estimates a
slow *envelope*. The threshold is the detector: low activation versus high.

I will not assume UCI 1 and 2 are “rest versus fist” until `prepare.py` has
checked the official names **and** the real RMS. The pair lives in
`meta["binary_pair"]`. If rest is not clearly quieter, I will call it an
amplitude-based binary problem. I will not force the rest-versus-fist story.


In [ ]:
print("Verified pair kind:", pair["kind"])
print("low:", pair["low_name"], "RMS", f"{pair['low_rms']:.3e}")
print("high:", pair["high_name"], "RMS", f"{pair['high_rms']:.3e}")
time_s, emg, raw_lab, truth = extract_binary_span(
    rec, pair["low_raw_label"], pair["high_raw_label"]
)
env = mean_channel_envelope(emg, win=20, method="mav")
thr = 0.5 * (env[truth == 0].mean() + env[truth == 1].mean())
pred = threshold_predict(env, thr)
acc = binary_accuracy(pred, truth)
print(f"chosen threshold {thr:.4f}  binary accuracy {acc:.3f}")
fig = plot_threshold_controller(
    time_s, emg[:, 0], env, thr, pred, truth,
    title=f"{pair_title}  (accuracy={acc:.2f})",
)
plt.show()


In [ ]:
thrs = np.linspace(float(env.min()), float(env.max()), 25)
accs = [binary_accuracy(threshold_predict(env, t_), truth) for t_ in thrs]
fig = plot_threshold_sweep(thrs, accs, title="Same recording, different thresholds")
plt.show()

wins = [5, 10, 20, 40, 80]
wacc = []
for w in wins:
    e = mean_channel_envelope(emg, win=w, method="mav")
    t_ = 0.5 * (e[truth == 0].mean() + e[truth == 1].mean())
    wacc.append(binary_accuracy(threshold_predict(e, t_), truth))
fig = plot_window_length_sweep(wins, wacc)
plt.show()
print("window lengths", wins)
print("accuracies   ", [round(a, 3) for a in wacc])

fig = plot_moving_average_as_lowpass(
    time_s, emg[:, 0], windows=[5, 20, 80], fs=fs, truth=truth,
    title="What averaging does: one rectified channel, three windows",
)
plt.show()


I want your eyes on three things.

The three-row figure: raw signal, envelope, and the red line on top; the
controller’s 0/1 in the middle; the true labels at the bottom. How well the
middle matches the bottom is how honest that switch is.

The second figure sweeps the threshold on the *same* recording. Raise the line,
lower the line, the number moves. So a threshold is tuned. It is not a physical
constant.

The third figure is window length. I also added an overlay — the same rectified
channel with 5 / 20 / 80-sample averages, plus the sinc frequency response of a
uniform window. I want you to *see* it: **averaging is a low-pass.** Longer
window, blunter edges, more lag, narrower passband.

Smoothing is not free. You buy a prettier envelope. You pay response time.
The “best” threshold on this file need not survive a new electrode, sweat, or
a new person.

Tell me: why does a bigger window look smoother? If the hand has already
closed, what happens to the switch?


## Experiment 3 — When amplitude is a mess, I take you to frequency

If the time plot is chaos, can the spectrum tell us *what kind* of disturbance
we are looking at?

Different physics lives in different bands. Instrumentation can be a needle.
EMG — and a lot of physiological crosstalk — is usually broadband.

I will plot the **observed** spectrum first. If there is no 50 Hz spike, I will
not tell you “see, that is mains.” No evidence, no claim of a 50 Hz artefact
in this file.

Then I will add a sine on purpose and label it **SYNTHETIC**. That second
figure is for teaching the notch. It is not something I found hiding in the
data.


In [ ]:
high_idx = np.where(raw_lab == pair["high_raw_label"])[0]
a = int(high_idx[0]); z = min(len(emg), a + int(2 * fs))
x = emg[a:z, 0]
tt = np.arange(len(x)) / fs
freq, mag = fft_magnitude(x, fs)
pf, psd = welch_psd(x, fs)
print("OBSERVED peak near 50 Hz (FFT magnitude):", round(peak_near(freq, mag, 50), 6))
fig = plot_time_fft_psd(
    tt, x, freq, mag, pf, psd,
    title="OBSERVED EMG window — no claim of 50 Hz artefact",
)
plt.show()


In [ ]:
x_syn = add_synthetic_mains(x, fs, freq=50.0)
f2, m2 = fft_magnitude(x_syn, fs)
p2f, p2 = welch_psd(x_syn, fs)
fig = plot_time_fft_psd(
    tt, x_syn, f2, m2, p2f, p2,
    title="SYNTHETIC teaching example:  x_real(t) + A sin(2π·50·t)",
    mark_hz=50.0, mark_label="SYNTHETIC 50 Hz",
)
plt.show()


If the unwanted piece occupies a very narrow band, which kind of filter do you
reach for? Think, then run.


## Experiment 4 — Filtering will help you. It will not finish the job

Can I filter my way out? A few minutes. I want the boundary to be obvious.

Filtering feels great when wanted and unwanted sit in different frequency
neighbourhoods. Crosstalk from the muscle next door often sits in the **same**
band as the EMG you want. Then there is no perfect pair of scissors.


In [ ]:
x_notch = notch_filter(x_syn, fs, freq=50.0)
pf_b, psd_b = welch_psd(x_syn, fs)
pf_a, psd_a = welch_psd(x_notch, fs)
fig = plot_notch_before_after(
    tt, x_syn, x_notch, pf_b, psd_b, pf_a, psd_a,
    title="Good: a SYNTHETIC narrow line can be notched",
)
plt.show()
fig = plot_crosstalk_diagram()
plt.show()


On the synthetic 50 Hz, the notch wins. The line is spectrally separate, so we
can cut it.

Now I change the assumption. Look at this:

\[
\text{target EMG spectrum} \;\cap\; \text{crosstalk spectrum} \;\neq\; \emptyset
\]

If two muscles overlap in frequency, what cutoff perfectly separates them?

My answer: **there may be no such cutoff.** Filtering stops here. The question
changes from separating signals to recognising a pattern.


---
# Act II — Then I let the machine read the pattern

Instead of dreaming we can isolate every muscle, I ask something else: is there
usable information in the *shape* of the multichannel mixture?


## Experiment 5 — Features you can still explain

I stop asking “did we cross the line?” I ask: “does this bundle of measurements
belong to this movement?”

RMS, mean absolute value, waveform length, zero crossings — summaries an
engineer can defend. Eight channels × four numbers: one 32-D vector per window.
The model **only sees those 32 numbers**, not the raw voltage trace.

Kill the name collision first:

This is **not** least-squares linear regression. That predicts one continuous
number (how hard did you squeeze?). The code is `StandardScaler` +
`LogisticRegression`: **multinomial logistic regression**, a softmax linear
classifier.

For each gesture \(k\) (rest, fist, wrist flexion, … — six classes) it learns
one linear score:

$$
s_k = w_k\cdot \tilde{x} + b_k
$$

\(\tilde{x}\) is the 32 numbers after subtracting the training mean and
dividing by the training standard deviation. Skip that and RMS versus
zero-crossings live on different planets; the weights become unreadable.

Six scores go through softmax (probabilities that sum to one). The prediction
is \(\arg\max_k s_k\). What it learns is six pairs \((w_k, b_k)\), about
\(6\times 32 + 6 = 198\) numbers. **No hidden layer. No neural net.** Versus
your Experiment 2 threshold: one number crossing a line versus six hyperplanes
in 32-D. That is why it can do six gestures, not only rest versus fist.

I will **deliberately** do a random window split first. The number will
probably look great. Then I will split by person. Do not fall in love with the
first number.


In [ ]:
names = [class_names[i] for i in sorted(np.unique(y))]
rand_tr, rand_te = random_window_split(y, test_size=0.25, seed=SEED)
print("random split sizes", len(rand_tr), len(rand_te))
print("subjects overlapping train/test:", sorted(int(s) for s in overlapping_subjects(subject, rand_tr, rand_te)))
rand_f = fit_eval(F[rand_tr], y[rand_tr], F[rand_te], y[rand_te], seed=SEED, max_iter=MAX_ITER)
W = rand_f["clf"].named_steps["clf"].coef_
b = rand_f["clf"].named_steps["clf"].intercept_
print("handcrafted dim per window", F.shape[1], "(8 ch × 4)")
print("weight matrix W", W.shape, "(classes × features); bias", b.shape)
print("trainable parameters", rand_f["n_probe_params"], "(entries of W + bias)")
print("rule: score s = W x_scaled + b, then argmax")
print("handcrafted + linear, RANDOM split accuracy:", round(rand_f["accuracy"], 3))
fig = plot_confusion(rand_f["confusion"], names, f"Random window split  acc={rand_f['accuracy']:.3f}")
plt.show()


Look at the overlap list. If the same person is in train *and* test, 97% has
**not** shown that this works on the next stranger.

A benchmark only means something when you know **what was held out**.


In [ ]:
sub_tr, sub_te = subject_split_indices(subject, train_subjects, test_subjects)
print("train subjects", train_subjects)
print("test  subjects", test_subjects)
print("n train windows", len(sub_tr), "n test windows", len(sub_te))
if len(sub_tr) == 0 or len(sub_te) == 0:
    print("Not enough held-out subjects in this FAST_DEMO subset.")
    sub_f = None
else:
    sub_f = fit_eval(F[sub_tr], y[sub_tr], F[sub_te], y[sub_te], seed=SEED, max_iter=MAX_ITER)
    print("handcrafted + linear, SUBJECT split accuracy:", round(sub_f["accuracy"], 3))
    fig = plot_confusion(sub_f["confusion"], names, f"Subject-wise split  acc={sub_f['accuracy']:.3f}")
    plt.show()


What did the machine learn here that your fixed threshold never saw? Say it in
your own words.


## Experiment 6 — I freeze TinyMyo. I only ask if the representation is usable

Before it ever saw *these* UCI labels, did TinyMyo pretraining already give us
a space where gestures are roughly linearly separable?

Listen: this is **not** fine-tuning. I freeze the encoder. I only train a
linear classifier on the embeddings.

I use: `pretraining/TinyMyo/TinyMyo.safetensors`  
I do not use: `UCI_EMG/UCI_finetune_5sec.safetensors`


In [ ]:
ckpt = default_checkpoint_path()
model = build_encoder(img_size=window_samples, in_chans=8, device=DEVICE)
load_report = load_pretrained_weights(model, ckpt)
model.to(DEVICE)
params = count_parameters(model)
print("checkpoint", ckpt)
print("DEVICE", DEVICE, "| encoder param device", next(model.parameters()).device)
print("missing keys (expected if head differs):", load_report["missing_keys"][:8])
print("encoder params:", params)
print(f"Raw input shape  [{len(X)}, 8, {window_samples}]")
print(f"Frozen encoder -> embedding shape {tuple(Z.shape)}  (dim={Z.shape[1]})")
smoke = encode_windows(model, X[:1], batch_size=1, device=DEVICE)
print("one-window encode on", DEVICE, "->", smoke.shape)
print("We are reusing cached embeddings for the class metrics; the encoder is not being trained.")


I want you to see that the meaning of the coordinates changed.

Raw \(X\) is a table of voltages: this electrode, this instant.

TinyMyo hands you

$$
z = [z_1, z_2, \ldots, z_{1536}]
$$

usually `8 × 192`. Those numbers are **not** samples. They are combinations of
time structure and cross-channel structure that helped the pretraining job
(masked patch reconstruction). They were not learned for today’s gesture names.

If a linear classifier works on \(z\), that only means movement-related
information became easier to pull out. That is all a linear probe is asking.


In [ ]:
rand_z = fit_eval(Z[rand_tr], y[rand_tr], Z[rand_te], y[rand_te], seed=SEED, max_iter=MAX_ITER)
print("trainable probe params", rand_z["n_probe_params"])
print("frozen encoder params ", params["frozen"])
print("linear probe, RANDOM split accuracy:", round(rand_z["accuracy"], 3))
if sub_f is not None:
    sub_z = fit_eval(Z[sub_tr], y[sub_tr], Z[sub_te], y[sub_te], seed=SEED, max_iter=MAX_ITER)
    print("linear probe, SUBJECT split accuracy:", round(sub_z["accuracy"], 3))
    fig = plot_confusion(sub_z["confusion"], names, f"TinyMyo probe, subject split acc={sub_z['accuracy']:.3f}")
    plt.show()
else:
    sub_z = None


The encoder was not trained to sit this exam. I am only asking whether the
space it learned while pretraining already makes the later classes easier to
cut.


## Experiment 7 — I will show you PCA. I will not do t-SNE magic

Which representation *looks* easier to cut?

A prettier PCA cloud is **not** evidence. The comparison is the classification
number. PCA is a linear cartoon. I put it here so you have a picture, not a
verdict.


In [ ]:
pca_f, ev_f = pca_2d(F, seed=SEED)
pca_z, ev_z = pca_2d(Z, seed=SEED)
print("handcrafted PCA variance ratio", np.round(ev_f, 3))
print("TinyMyo     PCA variance ratio", np.round(ev_z, 3))
note = "Visual intuition only; the comparison is the classification numbers."
fig = plot_pca_scatter(pca_f, y, class_names, subject, title="PCA of handcrafted features", note=note)
plt.show()
fig = plot_pca_scatter(pca_z, y, class_names, subject, title="PCA of frozen TinyMyo embeddings", note=note)
plt.show()


Which side looks easier to separate? Say it, then check the numbers. If they
disagree, believe the numbers.


---
# Act III — Does it survive a new person?

A laboratory classifier is not a prosthetic controller. Sit up for this act.


## Experiment 8 — When I give you almost no labels

If both sides only get a handful of labels, does pretraining become more
valuable?

This is my first check of **data efficiency**. Useful. Not the hardest question
I will ask you today.


In [ ]:
train_mask = np.isin(subject, train_subjects)
test_mask = np.isin(subject, test_subjects)
if test_mask.sum() == 0:
    idx_local = np.where(train_mask)[0]
    tr_i, te_i = random_window_split(y[train_mask], test_size=0.3, seed=SEED)
    tr_idx, te_idx = idx_local[tr_i], idx_local[te_i]
    few_note = "same people in train and test (only label count changes)"
else:
    tr_idx, te_idx = np.where(train_mask)[0], np.where(test_mask)[0]
    few_note = "labelled windows from seen people; test on unseen people"
print(few_note)
print("train windows", len(tr_idx), "test windows", len(te_idx))
fractions = [1.0, 0.25, 0.10]
if len(tr_idx) >= 80:
    fractions.append(0.05)
seeds = [0, 1] if FAST_DEMO else [0, 1, 2]
curve_f = few_label_curve(F[tr_idx], y[tr_idx], F[te_idx], y[te_idx], fractions, seeds, max_iter=MAX_ITER)
curve_z = few_label_curve(Z[tr_idx], y[tr_idx], Z[te_idx], y[te_idx], fractions, seeds, max_iter=MAX_ITER)
for name, curve in [("handcrafted", curve_f), ("TinyMyo probe", curve_z)]:
    print(name)
    for frac, stats in curve.items():
        print(f"  {100*frac:3.0f}% labels  {stats['mean']:.3f} ± {stats['std']:.3f}")
fig = plot_few_label_curves(
    {"handcrafted + linear": curve_f, "frozen TinyMyo + linear probe": curve_z},
    title=f"Limited labelled data — {few_note}",
)
plt.show()


Useful, yes. Still not the hardest part.

If the foundation-model probe **loses**, I will list the honest reasons with
you: the UCI task is not savage; pretraining is 2 kHz / 16 channels and we are
at 200 Hz / 8; the preprocessing is different; a linear probe cannot move the
encoder; handcrafted features are already strong on static gestures.

I will not change the dataset to manufacture a win. If it loses, it stays on
the table.


## Experiment 9 — Hold out a person (this is the climax I planned)

The gesture name did not change. Why does the distribution change when the
person does?

$$
\boxed{\text{Train on people you have seen}}
\quad\rightarrow\quad
\boxed{\text{Test on a person you have never seen}}
$$

If you are building a prosthetic controller, this is the exam that looks like
life. Anatomy, recruitment, electrode placement, skin impedance, recording
conditions, crosstalk — all move. The label still says “fist”. The signal is
not the same object.


In [ ]:
if sub_f is None or sub_z is None:
    print("Held-out-subject experiment not available on this subset.")
    print("That is a data limitation, not a hidden result.")
else:
    print("handcrafted, unseen subjects:", round(sub_f["accuracy"], 3))
    print("TinyMyo probe, unseen subjects:", round(sub_z["accuracy"], 3))
    fig = plot_grouped_bars(
        ["held-out subjects"],
        {"handcrafted": [sub_f["accuracy"]], "TinyMyo probe": [sub_z["accuracy"]]},
        ylabel="accuracy",
        title="Train on seen people, test on unseen people",
    )
    plt.show()


Why is this so much harder for a prosthesis than randomly slicing one person’s
windows? If you can say that clearly, today was worth it.


## Experiment 10 — Why I even mentioned foundation models — and why they are not magic

A foundation model is not “a bigger classifier” with a new name.

The shift I want you to take home:

**Before: learn each task from scratch. Now: learn a representation you can
reuse.**

It will not magically delete crosstalk. The real research question is whether
it can learn movement-useful structure that still stands after you change the
user and the recording.

### What they actually trained: cover a stretch of voltage, then put it back

We did **not** retrain TinyMyo today. `TinyMyo.safetensors` is a finished
encoder. Pretraining was also **not** on the UCI 481 file in front of you —
that job used 2 kHz corpora (NinaPro DB6/DB7, EMG2Pose). The homework is
whatever `vendor/TinyMyo.py` and the paper say.

**The original signal** is one window of time-domain voltage: at most 16
channels, 1000 samples (0.5 s at 2 kHz). Band-pass 20–450 Hz, 50 Hz notch,
scale to \([-1,1]\). **No gesture labels.** It is not learning “is this a fist?”

**The cut:** each channel alone, length 20, stride 20. Twenty samples at
2 kHz = 10 ms. Sixteen channels → 800 tokens. Each scrap becomes 192 numbers,
then the Transformer.

**The mask:** about half the scraps are replaced by a learnable `[MASK]`.
What disappears is **tens of milliseconds of waveform**, not one sample, so
cheap interpolation is not enough — it has to use neighbouring time and the
other electrodes.

**What is restored:** the pretraining head is one linear layer. Each token
emits 20 voltages, aiming at that original scrap. Smooth L1: masked half
must come back carefully; visible half gets a small penalty (0.1) so the net
cannot just copy. Then that head is thrown away. We freeze the encoder. The
linear probe only asks whether gestures are easy to cut.

The first figure is a **schematic** I drew. Grey bands are `[MASK]` patches.
That is not TinyMyo’s real reconstruction on UCI.


In [ ]:
fig = plot_three_pipelines()
plt.show()
fig = plot_model_ab_table()
plt.show()
fig = plot_summary_progression(ROOT / "outputs" / "emg_to_foundation_model.png")
plt.show()
print("saved", ROOT / "outputs" / "emg_to_foundation_model.png")


### Inside a battery-powered prosthetic hand — which one would you put?

I do not have a gold-star answer. You have to hold accuracy, robustness,
transfer, latency, memory, power, reliability, and how painful calibration is
in the same hand.

**The best model on the leaderboard is not automatically the best engineering
system.**

The sentence I want you to leave with: a classifier that works in the lab does
not automatically make a prosthetic controller that works in the world.
